In [ ]:
1. pip install django djangorestframework pyjwt

2. INSTALLED_APPS = [
    ...
    "rest_framework",
    "accounts",
]

REST_FRAMEWORK = {
    "DEFAULT_AUTHENTICATION_CLASSES": [
        "accounts.authentication.CustomJWTAuthentication",
    ]
}
3. 🔐 Step 4 — Create JWT Helpers (encode + decode)

import jwt
from datetime import datetime, timedelta
from django.conf import settings

SECRET_KEY = settings.SECRET_KEY
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30

🛂 Step 5 — Create Custom Authentication Class
def create_jwt_token(user_id):
    expire = datetime.utcnow() + timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    payload = {
        "user_id": user_id,
        "exp": expire,
        "iat": datetime.utcnow(),
    }
    token = jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)
    return token


def verify_jwt_token(token):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        return payload
    except jwt.ExpiredSignatureError:
        return None
    except jwt.InvalidTokenError:
        return None

Step 5 — Create Custom Authentication Class

from rest_framework.authentication import BaseAuthentication
from rest_framework.exceptions import AuthenticationFailed
from django.contrib.auth.models import User
from .utils import verify_jwt_token


class CustomJWTAuthentication(BaseAuthentication):

    def authenticate(self, request):
        auth_header = request.headers.get("Authorization")

        if not auth_header:
            return None

        if not auth_header.startswith("Bearer "):
            raise AuthenticationFailed("Invalid token header")

        token = auth_header.split(" ")[1]
        payload = verify_jwt_token(token)

        if payload is None:
            raise AuthenticationFailed("Invalid or expired token")

        try:
            user = User.objects.get(id=payload["user_id"])
        except User.DoesNotExist:
            raise AuthenticationFailed("User not found")

        return (user, None)
        
Step 6 — Create Views (Login + Protected API)

In [ ]:
A JWT contains:

Header

Payload

Signature

PART–2: Structure of a JWT Token

HEADER.PAYLOAD.SIGNATURE

1️⃣ Header (algorithm info)
{
  "alg": "HS256",
  "typ": "JWT"
}

2️⃣ Payload (user info + expiry)
{
  "user_id": 10,
  "role": "admin",
  "iat": 1733875795,
  "exp": 1733876695
}

3️⃣ Signature (security)

This is the most important part.
    HMACSHA256(
    base64(header) + "." + base64(payload),
    SECRET_KEY
)


In [ ]:
🔥 PART–3: How JWT Works (Step-by-step)

Step 1: User logs in

        User sends username/password.
        
        Backend verifies → if correct → backend creates JWT:

        token = jwt.encode(payload, SECRET_KEY, algorithm="HS256")

Backend returns token to the client.

Step 2: Client stores JWT

    Front-end/mobile stores token in:
    
    ✔ LocalStorage
    ✔ SecureStore (mobile)
    ✔ HTTP-Only cookie (recommended)

Step 3: Client sends token in every request

Example request:
   GET /profile
   Authorization: Bearer <jwt_token>

Step 4: Server verifies the JWT

Verification steps inside backend:

    4.1 — Extract token
         Server reads:
           Authorization: Bearer xyz.abc.123
   4.2 — Decode the JWT (but NOT trust it yet)
        Server reads header + payload.
    4.3 — Verify Signature
    
    Backend recalculates:
           expected_signature = HMACSHA256(header + "." + payload, SECRET_KEY)


In [ ]:
🔒 PART–4: How JWT Verification Works Internally

Here is the exact verification logic:

import jwt

try:
    payload = jwt.decode(
        token,
        SECRET_KEY,
        algorithms=["HS256"]
    )
except jwt.ExpiredSignatureError:
    print("TOKEN EXPIRED")
except jwt.InvalidTokenError:
    print("INVALID TOKEN")

The library performs:

Base64 decode header + payload
Calculate new signature
Compare with token's signature
Check expiry
Return payload

In [ ]:
🎯 PART–5: Why JWT Is Secure?

✔ You cannot modify the payload
→ signature mismatch
✔ You cannot create fake tokens
→ you need SECRET_KEY
✔ You cannot bypass expiry
→ exp check happens always

Security depends on one thing:
👉 Keep SECRET_KEY safe
If someone gets SECRET_KEY, they can create fake JWTs.

In [ ]:
Login → Backend generates JWT → Client stores token
       ↓
Client sends token with every request
       ↓
Backend verifies:
   - Is signature valid?
   - Is token expired?
   - Is user valid?
       ↓
If yes → Allow request
If no → Reject


In [ ]:
“JWT verification happens in three steps.
First, the server verifies the signature using the same secret key 
that was used to generate the token. This ensures the token is not modified.
Second, the server verifies the standard claims like exp, iat, nbf, 
and optionally iss or aud.
Third, it performs custom checks like whether the user exists or if
the token ID (jti) is blacklisted.
If all checks pass, the token is considered valid.”